<a href="https://colab.research.google.com/github/AdamClarkStandke/LLMs/blob/main/chapter8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reasoning Distillation

Reasoning Distillation is similar to [Knoweledge Distillation](https://en.wikipedia.org/wiki/Knowledge_distillation) in that a smaller LLM model i.e., **student** is trained on [reasoning traces](https://www.emergentmind.com/topics/reasoning-traces) of a larger LLM model i.e. **teacher**. Not only does Reasoning Distillation shrink LLM models so that they can run on phones, but interestingly, researchers have also found that reasoning performance can be improved through this training technique. As the book [Build a Reasoning Model (From Scratch)](https://sebastianraschka.com/books/) details:

> Reasoning performance can be improved not only through inference-time scaling and reinforcement learning, but also through distillation...Distillation is important because the strongest reasoning models are often too large and expensive to work with directly. For example, [DeepSeek-R1](https://arxiv.org/pdf/2501.12948) has 671 billion parameters. Systems at that scale are expensive to develop and deploy, and they are far beyond what most practitioners can run on local hardware. The team behind DeepSeek-R1 *created smaller model variants by distilling the 671-billion-parameter teacher model*

Currently there are three main ways to distill reasoning from teacher LLMs to student LLMs:
1. **Hard Distillation**
2. **Soft Distillation**  
3. **Hard Distillation with Soft Distillation**


**Hard Distillation** uses only the teacher-generated text as the training target for the student model. Namely, a teacher model *generates the reasoning traces and answers* and a student model is fine-tuned/trained to reproduce/copy them. Hard Distillation was implemented by the [book](https://sebastianraschka.com/books/) and hence this notebook. Also as a side note,[DeepSeek-R1](https://arxiv.org/pdf/2501.12948) used this same approach to distill their models.

**Soft Distillation** uses the teacher model's distribution of logit values to train the student model. As the book details: "this allows the student model to have *richer information about which alternative tokens the teacher model consdered plausible*." However, this technique is harder to implement, since most proprietary models do not share their distribution of logit values (hence implementation is generally done with [open-weight models](https://en.wikipedia.org/wiki/Open_weights))

Lastly, **Hard and Soft distilliation** are combined in which the student model is trained on the teachers generated output while also training on the teacher's distribution of logit values.



---



## Implementing Reasoning Distillation

Implementing Hard Distillation consists of 4 steps:
1. Generating a dataset for Reasoning Distillation
2. Preparing the dataset for Reasoning Distillation
3. Training a Pretrained-Model for Reasoning Distillation
4. Evaluate the Student model on MATH-500 Test Set

Step 1 deals with generating the dataset for Reasoning Distillation. As the [book](https://sebastianraschka.com/books/) details: "we take 12,000 problems from the [Mathematics Apptitude Test of Heuristics (MATH) dataset](https://openreview.net/pdf?id=7Bywt2mQsCe&utm_source=signaltonoise.beehiiv.com&utm_medium=referral&utm_campaign=openai-strawberry-and-what-bench) and feed these problems into an existing teacher model which in this case was [DeepSeek-R1](https://arxiv.org/pdf/2501.12948)." The answers produced by DeepSeek-R1 are colleced as training targets for the student model which in this case is the [Qwen3 0.6B base model](https://magazine.sebastianraschka.com/p/qwen3-from-scratch). As the [book](https://sebastianraschka.com/books/) details:

> Because generating teacher answers for all 12,000 math problems can be time and resource intensive, I created the dataset using the 671-billion-parameter DeepSeek-R1 model hosted via OpenRouter. *The full data generation cost was approximately $50 in API usage*...If you are curious about the data-generation process, the code and usage instructions are available in the [supplementary materials](https://github.com/rasbt/reasoning-from-scratch/tree/main/ch08/02_generate_distillation_data)

This notebook does not do a seperate [data-generation process](https://github.com/rasbt/reasoning-from-scratch/tree/main/ch08/02_generate_distillation_data) but uses the books' distilled dataset generated by DeepSeek-R1 as found at [HuggingFace](https://huggingface.co/datasets/rasbt/math_distill).


In [1]:
!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

Cloning into 'reasoning-from-scratch'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (220/220), done.
remote: Total 269 (delta 52), reused 148 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (269/269), 1.77 MiB | 4.99 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [2]:
import os
os.chdir('/content/reasoning-from-scratch/')
import json
import requests
import inspect
import random
from pathlib import Path
from pprint import pprint
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch08 import load_distill_data, format_distilled_answer, load_reasoning_tokenizer, build_examples, filter_examples_by_max_len
from reasoning_from_scratch.ch07 import download_from_github
from reasoning_from_scratch.qwen3 import download_qwen3_small, Qwen3Tokenizer

In [ ]:
#Loading the raw dataset
math_train = load_distill_data()
print("Dataset size:", len(math_train))

deepseek-r1-math-train.json: 107538.0 KB
Dataset size: 12000


In [ ]:
#Printing sample of formatted answer from math dataset
print(format_distilled_answer(math_train[4]))

<think>Okay, let's see. Sam was hired for 20 days. Each day he works, he earns $60, but if he doesn't work a day, they deduct $30 from his earnings. At the end of the 20 days, he got $660. I need to find out how many days he didn't work.

Hmm, so let me think. Let's break this down. Let's say he worked x days and didn't work y days. So total days x + y = 20. That's straightforward. Now, his earnings. For each day worked, he gets $60, so total earnings from working would be 60x. But for each day he didn't work, they subtract $30. Wait, how exactly does that subtraction work? Is it that he loses $30 per day not worked from his total earnings?

Wait, maybe. So if he didn't work y days, then he loses 30*y dollars. But wait, does he get a base salary and then deductions? Or is his total earnings 60x minus 30y?

The problem says, "On days that he works, he earns $60. For each day that he does not work, $30 is subtracted from his earnings." So probably, his total earnings are calculated by 60

In [ ]:
#Evaluating DeepSeek-R1 on MATH training dataset
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_json.py"
)

!uv run evaluate_json.py \
--json_path "deepseek-r1-math-train.json" \
--gtruth_answer gtruth_answer \
--generated_text message_content

evaluate_json.py: 3.3 KB
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Installed 161 packages in 259ms
Accuracy: 90.6% (10871/12000)


##  Step 2: Preparing the dataset for Reasoning Distillation

As the [book](https://sebastianraschka.com/books/) details at step 2, "we convert the raw dataset entries into training examples that can be consumed by the model. At a high level, this means formatting the prompts and answers, tokenizing them, and storing the resulting token IDs together with the prompt."

Stage 1 of this step consists of loading the [Qwen3 reasoning tokenizer](https://magazine.sebastianraschka.com/p/qwen3-from-scratch) because it supports the ```<think>...</think>``` tokens. The formated text that the [Qwen3 reasoning tokenizer](https://magazine.sebastianraschka.com/p/qwen3-from-scratch) returns is a wrapped prompt that when decoded looks as follows:
```
<|im_start|>user
Sam is hired for a 20-day period...<|im_end|>
<|im_start|>assistant
```
where ```<|im_start|>user``` marks the start of the user prompt,```<|im_end|>``` marks the end of the prompt, and ```<|im_start|>assistant``` marks the start of the model response.

However, for training,  **we need the full sequence of token IDs: the wrapped prompt, followed by the teacher answer, and then followed by an end-of sequence token**. Doing so will result in the following format for training:
```
<|im_start|>user
Sam is hired for a 20-day period...<|im_end|>
<|im_start|>assistant
<think>Okay, let me try to solve this problem...</think>\boxed{4}<|im_end|>

```
The function ```build_examples``` applies this formatting requirment to the raw training samples as found in the ```math_train``` dataset.

Lastly, becuase the teacher model outputs different sequence lengths for tokens when rendering an answer, we need to filter out answers whose sequence lengths are extreme outliers. To do so, we choose a maximum length for the sequence length and filter out token sequences that are greater than the maximum length. This notebook uses a maximum sequence length of 2048. Then the dataset is split into training and validation sets for training.  









In [ ]:
#Loading the Qwen3 reasoning tokenizer
tokenizer = load_reasoning_tokenizer()

tokenizer-reasoning.json: 100% (10 MiB / 10 MiB)


In [ ]:
#Formatting the dataset
examples, skipped = build_examples(math_train, tokenizer)

In [ ]:
print(f"Number of examples: {len(examples)}")
print(f"Number of skipped examples: {skipped}")

Number of examples: 12000
Number of skipped examples: 0


In [ ]:
#Decoded formatted training sample
print(tokenizer.decode(examples[4]['token_ids']))

<|im_start|>user
You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Sam is hired for a 20-day period. On days that he works, he earns $\$$60. For each day that he does not work, $\$$30 is subtracted from his earnings. At the end of the 20-day period, he received $\$$660. How many days did he not work?

Answer:<|im_end|>
<|im_start|>assistant
<think>Okay, let's see. Sam was hired for 20 days. Each day he works, he earns $60, but if he doesn't work a day, they deduct $30 from his earnings. At the end of the 20 days, he got $660. I need to find out how many days he didn't work.

Hmm, so let me think. Let's break this down. Let's say he worked x days and didn't work y days. So total days x + y = 20. That's straightforward. Now, his earnings. For each day worked, he gets $60, so total earnings from working would be 60x. But for each day he didn't work, they subtract $30. Wait, how exactly does that subtraction work? I

In [ ]:
#Filtering the dataset by max sequence length
filtered_examples = filter_examples_by_max_len(examples, max_len=2048)

Original: 12000
Filtered: 6695
Removed: 5305


In [ ]:
#Splitting the dataset into training and validation sets
rng = random.Random(123)
rng.shuffle(filtered_examples)

train_examples = filtered_examples[25:]
val_examples = filtered_examples[:25]
print(f"Number of train examples: {len(train_examples)}")
print(f"Number of validation examples: {len(val_examples)}")

Number of train examples: 6670
Number of validation examples: 25


## Step 3: Training a Pretrained-Model for Reasoning Distillation

During step 3 we first load the [pretrained Qwen3 base model](https://magazine.sebastianraschka.com/p/qwen3-from-scratch).  


And unlike the training done in chapter 6 in which [reinforcement learning with verifible rewards](https://arxiv.org/pdf/2506.14245) (RLVR) with [Group Relative Policy Optimization](https://arxiv.org/pdf/2402.03300) (GRPO) was used to minimize the policy gradient (PG) loss i.e.,  $L_{PG}=-\frac{1}{N}\sum_{i=1}^{N}A_{i}\sum_{t=1}^{T_{i}}log\ p_{w}(y_{t}^{i}|y_{\lt t}^{i},x^{i}) $ where N is the number of rollouts, $y_{t}^{i}$ are the tokens of the i-th generated response of length $T_{i}$, $y_{\lt t}^{i}$ represents all previously generated tokens in the response, $x^{i}$ is the input prompt, and $A_{i}$ is the advantage of the full rollout (**side note:** in chapter 7 we used the [CLipped policy gradient loss](https://arxiv.org/pdf/1707.06347) (CPG) along with [kullback-leibler (KL) divergence](https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence) for the loss).

In Reasoning Distilliation we use the [cross-entropy loss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.functional.cross_entropy.html) i.e. $ L_{(CE)}=-\frac{1}{N}\sum_{N}\sum_{i=1}^{V}y_{i}log(p_{i})$ where N is the number of answer tokens, V is the vocabulatry of the model i.e., logit values, $y_{i}$ is the token that represents the answer target of the teacher model and $p_{i}$ is the predicted token of the student model. As the [book](https://sebastianraschka.com/books/) details we "only compute the cross-entropy as an **Answer-only loss** which means that the training loss is computed only on the teacher's answer tokens, not the prompt tokens."

The function ```train_distillation``` implements the training loop with the following 5 steps:
1. For each training epoch, shuffle the dataset
2. For each training step reset loss gradients, calculate cross-entropy loss, backpropogate and update model weights with loss greadients
3. Calculate validation loss and metrics
4. Save model checkpoint
5. Repeat step 1

I trained the the model for 2 epochs with a maximum sequence length of 2,048 tokens using only one [L4 GPU](https://www.nvidia.com/en-us/data-center/l4/) offered by Colab Pro. The training required about 19 GB of GPU memory and took only 3 hrs to train. Which compared to training the model using reinforcement learning with verifible rewards (RLVR) and Group Relative Policy Optimization (GRPO) is considered **light and cheap**. Unfortunately, I did not save the metrics compiled during training but I did save the model checkpoints and can be found and downloaded [here](https://huggingface.co/DexHeim/Qwen3_ReasoningDistillation_Math500/tree/main). Similar to the book I ended with a validation loss in the 0.6 range after 2 epochs.




In [ ]:
import torch
import time
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer
from reasoning_from_scratch.ch08 import train_distillation, compute_example_loss

In [ ]:
device = get_device()
model, _ = load_model_and_tokenizer(
    which_model="base",
    device = device,
    use_compile = False
)

Using NVIDIA CUDA GPU
qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [ ]:
#Un-comment to run the Training Loop
# torch.manual_seed(0)
# train_distillation(
#     model,
#     train_examples,
#     val_examples,
#     device,
#     epochs=2,
#     lr=5e-06,
#     grad_clip_norm=1.0,
#     seed=123,
#     log_every=5,
#     csv_log_path="train_distill_metrics.csv"
# )

[Epoch 1/2 Step 5/13340] train_loss=1.0859 val_loss=0.9167
[Epoch 1/2 Step 10/13340] train_loss=0.6797 val_loss=0.8964
[Epoch 1/2 Step 15/13340] train_loss=0.7227 val_loss=0.8811
[Epoch 1/2 Step 20/13340] train_loss=0.8984 val_loss=0.8694
[Epoch 1/2 Step 25/13340] train_loss=0.8906 val_loss=0.8602
[Epoch 1/2 Step 30/13340] train_loss=0.8047 val_loss=0.8514
[Epoch 1/2 Step 35/13340] train_loss=0.7812 val_loss=0.8455
[Epoch 1/2 Step 40/13340] train_loss=0.9258 val_loss=0.8395
[Epoch 1/2 Step 45/13340] train_loss=0.5664 val_loss=0.8333
[Epoch 1/2 Step 50/13340] train_loss=0.8164 val_loss=0.8275
[Epoch 1/2 Step 55/13340] train_loss=0.8242 val_loss=0.8237
[Epoch 1/2 Step 60/13340] train_loss=0.7812 val_loss=0.8177
[Epoch 1/2 Step 65/13340] train_loss=0.9297 val_loss=0.8147
[Epoch 1/2 Step 70/13340] train_loss=0.6836 val_loss=0.8108
[Epoch 1/2 Step 75/13340] train_loss=0.8594 val_loss=0.8075
[Epoch 1/2 Step 80/13340] train_loss=0.7305 val_loss=0.8050
[Epoch 1/2 Step 85/13340] train_loss=0.64

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

## 4. Evaluate the Student model on MATH-500 Test Set

Ths last step is testing the student model on the MATH-500 test set. The [official Qwen3 0.6B reasoning model](https://arxiv.org/pdf/2505.09388) i.e. table 19, achieves **77.6%** accuracy on the MATH-500 test set and the book's distillation achieves an accuracy of **33.6%** using the saved checkpoint after the 3rd epoch.

In [3]:
#Download the model checkpoints from my repo
!hf download DexHeim/Qwen3_ReasoningDistillation_Math500 --local-dir /Qwen3

Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/3.01G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/3.01G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.01G [00:00<?, ?B/s]

Fetching 4 files:  25% 1/4 [00:00<00:00,  3.50it/s]
Reconstructing (incomplete total...):   0% 21.0/3.01G [00:00<11322:25:04, 73.8B/s]

Reconstructing (incomplete total...):  86% 2.58G/3.01G [00:10<00:00, 525MB/s,  603MB/s  ]

Reconstructing (incomplete total...):  93% 2.81G/3.01G [00:

In [4]:
#Get the Evaluation script
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)

evaluate_math500.py: 3.5 KB


In [7]:
# !uv run evaluate_math500.py \
# --which_model reasoning \
# --max_new_tokens 2048 \
# --checkpoint_path \
# "/Qwen3/qwen3-0.6B-distill-step13340-epoch2.pth"